# Column Lineage End-to-End

This notebook demonstrates the lean `llm4lineage` flow:

1. **Deterministic parsing** with `sqlglot`
2. **Column-level lineage** via `sqlglot.lineage` (`ColumnLineageExtractor`)
3. **Optional schema catalog** to expand `SELECT *`
4. **SQL2Graph** column dependency graph (deterministic parser + structured LLM enrichment)

In [ ]:
from Classes import (
    Config,
    PipelineOrchestrator,
    SQL2GraphParser,
    SQLLogicalChunkParser,
    setup_logging,
)

setup_logging("INFO")

## 1. Deterministic column lineage

In [ ]:
SQL = """
WITH recent AS (
    SELECT customer_id, SUM(amount) AS total
    FROM orders
    GROUP BY customer_id
)
SELECT c.name, r.total
FROM customers c
JOIN recent r ON c.id = r.customer_id
"""

SCHEMA = {
    "customers": ["id", "name"],
    "orders": ["customer_id", "amount"],
}

config = Config(llm_provider="mock")
orchestrator = PipelineOrchestrator(config, schema_catalog=SCHEMA)
result = orchestrator.run(SQL, instruction="Summarize column dependencies.")

print("success:", result.success)
for row in result.column_lineage:
    print(row["target_column"], "<-", row["source_columns"])

## 2. SELECT * expansion with schema catalog

In [ ]:
star_result = orchestrator.run("SELECT * FROM customers")
print([row["target_column"] for row in star_result.column_lineage])

## 3. Deterministic SQL chunk split

In [ ]:
chunk_parser = SQLLogicalChunkParser()
chunks = chunk_parser.preparse(SQL)
print("chunks:", [c["id"] for c in chunks["chunks"]])
print("links:", chunks["links"])

## 4. SQL2Graph deterministic simplify payload

In [ ]:
parser = SQL2GraphParser()
simplified = parser.simplify(SQL)
print("column_lineage keys:", [c["target_column"] for c in simplified.get("column_lineage", [])])
print("from tables:", simplified.get("from"))